Homework for the course of Data Science
===

Done by: BARBAUD Malo, NASSIET-COMBE Anne-Amélie, DOVHAN Nikita
---

In this notebook we will explore different Large Language Models (LLMs) that are currently available on the market. In order to utilize them we will utilize an external OpenRouter SDK which will allow us to execute requests to models on third-party services rather than our laptops.

Goal of the homework: use the OpenRouter SDK to discover **cheap multimodal models** (models that accept images in addition to text) and compare a few of them by asking the same prompt about **software development, AI, big data, DevSecOps and SRE**, then compare their answers, latency, token usage and cost.

Prerequisites
===

In [1]:
import os
import time
from openrouter import OpenRouter
import dotenv

dotenv.load_dotenv("../.env")

True

In [2]:
router = OpenRouter(api_key=os.getenv("OPENROUTER_KEY"))
router.models.list()

GetModelsResponse(next=<function Models.list.<locals>.next_func at 0x7fdde69e3530>, result=ModelsListResponse(data=[Model(architecture=ModelArchitecture(input_modalities=['text'], modality='text->text', output_modalities=['text'], instruct_type=None, tokenizer='Other'), canonical_slug='ricursiveai/ripple-65b-preview-20261104', context_length=16384, created=1793818990, default_parameters=DefaultParameters(frequency_penalty=Unset(), presence_penalty=Unset(), repetition_penalty=Unset(), temperature=Unset(), top_k=Unset(), top_p=Unset()), endpoints=['/api/v1/chat/completions', '/api/v1/completions', '/api/v1/messages', '/api/v1/responses'], id='ricursiveai/ripple-65b-preview', links=ModelLinks(details='/api/v1/models/ricursiveai/ripple-65b-preview-20261104/endpoints'), name='ricursiveAI/Ripple-65B-Preview', per_request_limits=None, pricing=PublicPricing(completion='0.0000006', prompt='0.0000002', audio=None, audio_output=None, discount=None, image=None, image_output=None, image_token=None,

Step 1 — Discover cheap multimodal models on OpenRouter
===

The `models.list()` endpoint is paginated, so we walk through every page and collect all models. Then we keep only the ones that are:

* **multimodal**: their `architecture.input_modalities` contains `"image"` (they can see pictures, not just text),
* **cheap**: the prompt price is below a threshold we fix (here `$0.20 per million input tokens`), and it is strictly positive (we exclude the `:free` variants so that the comparison is done on real paid tiers),
* **usable for chat**: they must have at least one endpoint with a positive completion price.

Finally we sort the candidates by blended price (input + output) and keep the `N_MODELS` cheapest ones for the comparison.

In [3]:
N_MODELS = 4            # how many models we compare
MAX_PRICE_PER_M = 0.20  # max $ per million input tokens to be considered "cheap"

# Walk through all pages of the model catalog
all_models = []
page = router.models.list()
while page is not None:
    all_models.extend(page.result.data)
    page = page.next() if callable(getattr(page, "next", None)) else None

print(f"Total models available on OpenRouter: {len(all_models)}")

candidates = []
for m in all_models:
    if m.id.endswith(":batch"):   # batch variants cannot be used with chat/completions
        continue
    arch = getattr(m, "architecture", None)
    pricing = getattr(m, "pricing", None)
    if not arch or not arch.input_modalities or not pricing:
        continue
    if "image" not in arch.input_modalities:   # not multimodal
        continue
    try:
        in_price = float(pricing.prompt)        # $ per token
        out_price = float(pricing.completion)
    except (TypeError, ValueError):
        continue
    if not (0 < in_price * 1e6 <= MAX_PRICE_PER_M) or out_price <= 0:
        continue
    candidates.append({
        "id": m.id,
        "name": m.name,
        "context": m.context_length,
        "in_price_M": in_price * 1e6,
        "out_price_M": out_price * 1e6,
        "modalities": ",".join(arch.input_modalities),
    })

candidates.sort(key=lambda c: c["in_price_M"] + c["out_price_M"])
selected = candidates[:N_MODELS]

print(f"Cheap multimodal candidates found: {len(candidates)}\n")
print(f"{'Model ID':<45} {'ctx':>7} {'$/M in':>8} {'$/M out':>8}  modalities")
print("-" * 90)
for c in candidates[:15]:
    print(f"{c['id']:<45} {c['context']:>7} {c['in_price_M']:>8.3f} {c['out_price_M']:>8.3f}  {c['modalities']}")
print("\nSelected for the comparison:")
for c in selected:
    print(f"  - {c['id']}")

Total models available on OpenRouter: 418
Cheap multimodal candidates found: 62

Model ID                                          ctx   $/M in  $/M out  modalities
------------------------------------------------------------------------------------------
nex-agi/nex-n2.5-mini                          262144    0.025    0.100  text,image
google/gemma-3-4b-it                           131072    0.050    0.100  text,image
qwen/qwen3.7-flash                            1000000    0.030    0.130  text,image,video
rekaai/reka-edge                                16384    0.100    0.100  image,text,video
mistralai/ministral-3b-2512                    131072    0.100    0.100  text,image
google/gemma-3-12b-it                          131072    0.050    0.150  text,image
prism-ml/ternary-bonsai-2-9b                   262144    0.080    0.130  text,image
google/gemma-4-26b-a4b-it                      262144    0.055    0.164  image,text,video
inclusionai/ling-3.0-flash-vl                  131072 

Step 2 — Build the comparison prompt
===

Every model receives **exactly the same prompt**. The prompt asks one short question about each of the five topics we study: **software development, AI, big data, DevSecOps and SRE**. We ask for concise answers so that the models stay comparable in terms of output length (and therefore cost).

In [4]:
PROMPT = """You are a senior engineer mentoring a Data Science student.
Answer each of the following five questions in 2-3 sentences, numbering your answers 1 to 5:

1. Software development: What are the main advantages of writing unit tests and using continuous integration in a software project?
2. AI: What is the difference between supervised learning and reinforcement learning? Give one concrete example of each.
3. Big Data: Explain the "3 V's" of big data and name one storage or processing technology suited to each V.
4. DevSecOps: How would you integrate security checks (secrets scanning, dependency auditing, SAST) into a CI/CD pipeline without slowing down deliveries?
5. SRE: What is the difference between an SLI, an SLO and an SLA, and how does an error budget help a team balance reliability and feature velocity?
"""
print(PROMPT)

You are a senior engineer mentoring a Data Science student.
Answer each of the following five questions in 2-3 sentences, numbering your answers 1 to 5:

1. Software development: What are the main advantages of writing unit tests and using continuous integration in a software project?
2. AI: What is the difference between supervised learning and reinforcement learning? Give one concrete example of each.
3. Big Data: Explain the "3 V's" of big data and name one storage or processing technology suited to each V.
4. DevSecOps: How would you integrate security checks (secrets scanning, dependency auditing, SAST) into a CI/CD pipeline without slowing down deliveries?
5. SRE: What is the difference between an SLI, an SLO and an SLA, and how does an error budget help a team balance reliability and feature velocity?



Step 3 — Query every selected model and collect metrics
===

For each model we call `router.chat.send()` with the same prompt and we record:

* the **wall-clock latency** of the call,
* the **token usage** (`prompt_tokens`, `completion_tokens`) and the **cost** reported by OpenRouter in `usage.cost`,
* the generated **answer** itself.

Requests are sent sequentially so that the latency measurements are not polluted by concurrency. A failure on one model (rate limit, provider error...) does not stop the comparison: it is simply recorded as an error.

In [5]:
results = []

for c in selected:
    model_id = c["id"]
    print(f"Querying {model_id} ...", end=" ", flush=True)
    start = time.perf_counter()
    try:
        response = router.chat.send(
            model=model_id,
            messages=[{"role": "user", "content": PROMPT}],
            max_tokens=800,
            temperature=0.2,
        )
        latency = time.perf_counter() - start
        usage = response.usage
        cost = usage.cost or 0.0
        results.append({
            "model": model_id,
            "ok": True,
            "latency_s": round(latency, 2),
            "prompt_tokens": usage.prompt_tokens,
            "completion_tokens": usage.completion_tokens,
            "cost_usd": cost,
            # some models return content=None and put the text in `reasoning` instead
            "answer": (response.choices[0].message.content
                       or response.choices[0].message.reasoning
                       or "(empty answer)"),
        })
        print(f"done in {latency:.1f}s ({usage.completion_tokens} output tokens, ${cost:.6f})")
    except Exception as e:
        latency = time.perf_counter() - start
        results.append({
            "model": model_id,
            "ok": False,
            "latency_s": round(latency, 2),
            "prompt_tokens": None,
            "completion_tokens": None,
            "cost_usd": None,
            "answer": f"ERROR: {type(e).__name__}: {e}",
        })
        print("failed")

Querying nex-agi/nex-n2.5-mini ... done in 3.1s (453 output tokens, $0.000047)
Querying google/gemma-3-4b-it ... done in 14.4s (512 output tokens, $0.000033)
Querying qwen/qwen3.7-flash ... done in 10.3s (800 output tokens, $0.000110)
Querying rekaai/reka-edge ... done in 2.3s (218 output tokens, $0.000040)


Step 4 — Quantitative comparison
===

We summarize the run with a small table: price per million tokens (from the catalog), measured latency, tokens consumed, cost of this single prompt, and a rough throughput in output tokens per second.

In [6]:
prices = {c["id"]: c for c in selected}

header = (f"{'Model':<40} {'$/M in':>7} {'$/M out':>8} {'Latency':>8} "
          f"{'in tok':>7} {'out tok':>8} {'Cost':>10} {'tok/s':>7}")
print(header)
print("-" * len(header))
for r in results:
    p = prices[r["model"]]
    if r["ok"]:
        tps = r["completion_tokens"] / r["latency_s"] if r["latency_s"] else 0
        print(f"{r['model']:<40} {p['in_price_M']:>7.3f} {p['out_price_M']:>8.3f} "
              f"{r['latency_s']:>7.2f}s {r['prompt_tokens']:>7} {r['completion_tokens']:>8} "
              f"${r['cost_usd']:>9.6f} {tps:>7.1f}")
    else:
        print(f"{r['model']:<40} {p['in_price_M']:>7.3f} {p['out_price_M']:>8.3f} "
              f"{r['latency_s']:>7.2f}s {'-':>7} {'-':>8} {'-':>10} {'FAILED':>7}")

ok = [r for r in results if r["ok"]]
if ok:
    total = sum(r["cost_usd"] for r in ok)
    cheapest = min(ok, key=lambda r: r["cost_usd"])
    fastest = min(ok, key=lambda r: r["latency_s"])
    print(f"\nTotal cost of the comparison run: ${total:.6f}")
    print(f"Cheapest answer : {cheapest['model']} (${cheapest['cost_usd']:.6f})")
    print(f"Fastest answer  : {fastest['model']} ({fastest['latency_s']:.2f}s)")

Model                                     $/M in  $/M out  Latency  in tok  out tok       Cost   tok/s
------------------------------------------------------------------------------------------------------
nex-agi/nex-n2.5-mini                      0.025    0.100    3.09s     192      453 $ 0.000047   146.6
google/gemma-3-4b-it                       0.050    0.100   14.44s     190      512 $ 0.000033    35.5
qwen/qwen3.7-flash                         0.030    0.130   10.34s     193      800 $ 0.000110    77.4
rekaai/reka-edge                           0.100    0.100    2.25s     182      218 $ 0.000040    96.9

Total cost of the comparison run: $0.000230
Cheapest answer : google/gemma-3-4b-it ($0.000033)
Fastest answer  : rekaai/reka-edge (2.25s)


Step 5 — Qualitative comparison of the answers
===

Numbers only tell half of the story: we print the full answer of each model to judge the quality on the five topics (software development, AI, big data, DevSecOps, SRE). Read them and ask yourself: did the model answer all five questions? Is the content correct? Is it concrete or vague?

In [7]:
for r in results:
    print("=" * 100)
    print(f"MODEL: {r['model']}")
    print("=" * 100)
    print(r["answer"].strip())
    print()

MODEL: nex-agi/nex-n2.5-mini
1. **Software development:** Unit tests give fast feedback, protect refactoring, and catch regressions early. Continuous integration ensures everyone’s changes are automatically built, tested, and validated before merging, reducing “it works on my machine” problems.

2. **AI:** Supervised learning learns from labeled input-output examples, such as classifying emails as spam or not spam. Reinforcement learning learns by taking actions in an environment to maximize reward, such as a robot learning to walk or an AI agent learning to play chess.

3. **Big Data:** The “3 V’s” are **Volume** — large amounts of data, suited to HDFS or object storage; **Velocity** — fast data generation or processing, suited to Kafka or Spark Streaming; and **Variety** — structured, semi-structured, and unstructured data, suited to NoSQL databases like MongoDB or Cassandra.

4. **DevSecOps:** Run lightweight security checks automatically in CI/CD: secrets scanning on every pull req

Conclusion
===

After running the four cheapest multimodal models on the same five-topic prompt, we draw the following observations:

### Best overall answer quality

**nex-agi/nex-n2.5-mini** delivered the most complete and accurate answers across all five topics. Every response was well-structured, concise (2–3 sentences as requested), and factually correct — naming appropriate technologies for each Big-Data "V", clearly distinguishing supervised from reinforcement learning, and correctly defining SLI / SLO / SLA / error-budget. **google/gemma-3-4b-it** was a close second with equally accurate content, though slightly more verbose (it added an unsolicited follow-up question at the end).

**qwen/qwen3.7-flash** produced accurate draft answers but leaked its internal chain-of-thought / scratchpad into the output, making the response much harder to read and consuming 800 output tokens — nearly double the others. **rekaai/reka-edge** was the weakest: it incorrectly named the 3 V's as "Storage, Processing, Visualization", confused the SLI/SLO/SLA definitions (e.g. "SLI measures service level objectives"), and gave a notably superficial answer overall.

### Best quality / price ratio

| Model | Cost | Quality | Verdict |
| --- | --- | --- | --- |
| google/gemma-3-4b-it | **$0.000033** (cheapest) | Very good | 🏆 Best quality/price |
| nex-agi/nex-n2.5-mini | $0.000047 | Excellent | Best pure quality |
| rekaai/reka-edge | $0.000040 | Poor | Low value despite low price |
| qwen/qwen3.7-flash | $0.000110 (most expensive) | Good (noisy) | Worst quality/price |

**google/gemma-3-4b-it** wins the quality/price trade-off: it costs the least of all four and still produces accurate, well-formatted answers.

### Best latency

**rekaai/reka-edge** was the fastest at **2.25 s**, followed by nex-n2.5-mini at 3.09 s. However, reka-edge achieved that speed partly because it generated the fewest tokens (218) — and those tokens were the least accurate. nex-n2.5-mini offers the best **throughput** at **146.6 tok/s**, making it the best choice when both speed and quality matter.

### Is the cheapest model good enough for a student use case?

Yes. **google/gemma-3-4b-it** at $0.000033 per prompt produced answers that are accurate and detailed enough to help a student review or study these five topics. At this price a student could run hundreds of thousands of such queries for less than a dollar — cost is effectively not a concern.

### Possible follow-up

Since all four selected models accept images alongside text, a natural next step would be to send the same prompt together with an architecture diagram or a monitoring dashboard screenshot and compare how well each model interprets the visual context — testing their true multimodal capabilities rather than text-only performance.
